# 토지피복의 면적은 어떻게 계산하는가?

범주형 지도에서 면적 가중 집계 이해

**할 일** · WorldCover의 범주별 면적과 물·도시·수목 비율 계산

위에서 아래로 실행. 코드를 바꾸기 전 결과를 예상하고, 실행 결과와 비교

## 실행 준비

CPU 런타임 사용. Earth Engine에 등록한 Cloud 프로젝트와 사용 권한 필요. [설정 안내](https://github.com/steelpl/lecture/blob/main/CE776-climate-change-carbon-water-cycle/GEE_SETUP.md) 참고. 데이터 조회에는 본인 계정 인증 필요

In [ ]:
%pip -q install earthengine-api==1.7.52 geemap==0.39.0

In [ ]:
import ee
import geemap

PROJECT_ID = ""  # Earth Engine에 등록한 본인의 Cloud 프로젝트 ID
if not PROJECT_ID.strip():
    raise ValueError("PROJECT_ID에 본인의 등록된 Cloud 프로젝트 ID 입력")
ee.Authenticate()
ee.Initialize(project=PROJECT_ID)

In [ ]:
roi = ee.Geometry.Rectangle([126.80, 37.43, 127.15, 37.70], geodesic=False)

In [ ]:
land = ee.ImageCollection('ESA/WorldCover/v100').first().select('Map').clip(roi)
area_image = ee.Image.pixelArea().divide(1e6).rename('area_km2').addBands(land)
areas = area_image.reduceRegion(ee.Reducer.sum().group(groupField=1,groupName='class'),roi,10,maxPixels=20_000_000)

In [ ]:
groups = areas.getInfo().get('groups',[])
class_names = {10:'Tree cover',20:'Shrubland',30:'Grassland',40:'Cropland',50:'Built-up',60:'Bare/sparse',70:'Snow/ice',80:'Permanent water',90:'Herbaceous wetland',95:'Mangroves',100:'Moss/lichen'}
total = sum(g['sum'] for g in groups)
for g in groups:
    print(class_names.get(g['class'],g['class']), round(g['sum'],3), 'km2', round(100*g['sum']/total,2), '%')

In [ ]:
Map = geemap.Map()
Map.centerObject(roi,10)
classes = [10,20,30,40,50,60,70,80,90,95,100]
palette = ['006400','ffbb22','ffff4c','f096ff','fa0000','b4b4b4','f0f0f0','0064c8','0096a0','00cf75','fae6a0']
Map.addLayer(land.remap(classes, list(range(11))), {'min':0,'max':10,'palette':palette}, 'WorldCover 2020')
Map

범주 번호를 평균 내면 토지피복을 나타낼 수 있는지 설명하시오. 분모가 전체 행정구역인지 유효하게 분류된 관심 영역인지 확인하시오.

**정리** · 범주는 순서 없는 이름. 면적은 `pixelArea`로 집계. 2020판과 2021판은 알고리즘 차이가 있어 단순 차이를 전부 실제 변화로 해석하기 어려움

## 자료와 해석 범위

- [WorldCover 2020 · ESA WorldCover Consortium · CC BY 4.0](https://developers.google.com/earth-engine/datasets/catalog/ESA_WorldCover_v100)

관측 연도는 고정 사례의 기간. 수업 연도와 무관. 코드는 자료 처리 방법을 보여 주며 지도·수치는 인증 후 조회 결과로 판단